# Notebook 01: — Curación de Datos de Mutaciones COSMIC para genes RAS

**Notebook:** `01_data_curation.ipynb`  
**TFM:** Bioinformática estructural — Familia RAS  
**Objetivo:** Cargar, filtrar y curar el catálogo de mutaciones somáticas de COSMIC para los genes KRAS, HRAS y NRAS, generando un dataset limpio listo para los hitos siguientes del pipeline.

---

## ¿Qué es COSMIC?

COSMIC (*Catalogue Of Somatic Mutations In Cancer*, https://cancer.sanger.ac.uk/cosmic) es la mayor base de datos pública de mutaciones somáticas en tumores humanos. La mantiene el Instituto Sanger (Cambridge, UK) y recopila información de miles de estudios de secuenciación del genoma del cáncer. Cada entrada de COSMIC registra una mutación encontrada en una muestra tumoral concreta, junto con metadatos clínicos como el tipo histológico del tumor y el tejido de origen.

## ¿Qué es una mutación missense?

Una **mutación missense** (o de sentido erróneo) es aquella que cambia un único nucleótido en el ADN de tal manera que el codón resultante codifica un aminoácido diferente al original. Por ejemplo, en KRAS la mutación G12D sustituye la Glicina (G) de la posición 12 por Ácido Aspártico (D). Este cambio aparentemente pequeño puede tener consecuencias funcionales drásticas: en el caso de los oncogenes RAS, muchas mutaciones missense bloquean la actividad GTPasa de la proteína, dejándola atrapada en estado activo y disparando señales de proliferación celular de forma incontrolada.

A diferencia de las mutaciones **silenciosas** (cambia el nucleótido pero no el aminoácido), las mutaciones **sin sentido** (generan un codón de parada prematuro) o las **inserciones/deleciones** (frameshift), las missense son las de mayor relevancia funcional para el estudio de oncogenes de ganancia de función como KRAS, HRAS y NRAS.

## ¿Por qué curamos los datos?

Los datos brutos de COSMIC contienen:
- Variantes **germinales** (heredadas, no somáticas adquiridas por el tumor): no son relevantes para el estudio de oncogénesis somática.
- Variantes de tipos no missense: no alteran la secuencia proteica de forma puntual.
- **Duplicados**: la misma mutación en la misma muestra puede aparecer más de una vez por reanálisis o múltiples publicaciones. Contarlos inflaría artificialmente las frecuencias.
- Genes que no son de interés (p.ej. BRAF, TP53): solo nos interesan los tres miembros de la familia RAS clásica.

El proceso de curación convierte datos crudos heterogéneos en un dataset estructurado, reproducible y listo para análisis estadístico y estructural.

## Contexto biológico: la familia RAS

Los genes **KRAS** (Kirsten RAS), **HRAS** (Harvey RAS) y **NRAS** (Neuroblastoma RAS) codifican GTPasas de pequeño tamaño (~21 kDa) que actúan como interruptores moleculares binarios:
- **Estado activo**: unidas a GTP, activan rutas de señalización como MAPK/ERK y PI3K/AKT, promoviendo la proliferación y supervivencia celular.
- **Estado inactivo**: unidas a GDP, no transmiten señal proliferativa.

La transición entre estados está regulada por proteínas GAP (que estimulan la hidrólisis de GTP a GDP) y GEF (que intercambian GDP por GTP). Las mutaciones en los hotspots **G12**, **G13** y **Q61** bloquean la actividad GTPasa y mantienen a la proteína en estado activo, constituyendo uno de los eventos oncogénicos más frecuentes en cáncer humano (~30% de todos los tumores).

---

## Contenido
1. Carga del fichero COSMIC en bruto
2. Filtrado: somáticas confirmadas, missense
3. Deduplicación por sample_id
4. Armonización de nomenclatura HGVS
5. Construcción del DataFrame curado
6. Descarga de PDBs y validación
7. Guardado de outputs en `data/processed/`

Lee antes `docs/primeros_pasos.md`, `docs/recursos_tecnicos.md` y `docs/contratos_datos.md`.

## 1. Importaciones y carga del fichero COSMIC

> **TODO (alumno):** primero usa el TSV de ejemplo para desarrollar. Cuando tengas COSMIC real, colocalo en `data/raw/cosmic/` y documenta la version exacta usada.

In [1]:
# ============================================================
# CHUNK 1.1: Importaciones y configuración general
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from Bio.PDB import PDBList, PDBParser
from functools import reduce

from tfm_ras.config import load_config, project_root
from tfm_ras import cosmic_loader

ROOT = project_root()

CFG = load_config(ROOT / 'configs' / 'config.yaml')
RAW = ROOT / CFG['paths']['data_raw']

EXAMPLE = ROOT / CFG['paths']['data_example']

OUT = ROOT / CFG['paths']['data_processed']
OUT.mkdir(parents=True, exist_ok=True)

np.random.seed(CFG['project']['seed'])

In [2]:
# ============================================================
# CHUNK 1.2: Carga del fichero de COSMIC
# ============================================================


raw_filename = (RAW / 'cosmic' 
                   / f"Cosmic_MutantCensus_Tsv_{CFG['cosmic']['version']}_{CFG['cosmic']['human_reference_genome']}" 
                   / CFG['cosmic']['raw_filename'].format(**CFG['cosmic']))

classification_filename = (RAW / 'cosmic' 
                             / f"Cosmic_Classification_Tsv_{CFG['cosmic']['version']}_{CFG['cosmic']['human_reference_genome']}" 
                             / CFG['cosmic']['classification_filename'].format(**CFG['cosmic']))

example_cosmic = EXAMPLE / CFG['cosmic']['example_filename']


if raw_filename.exists() and classification_filename.exists():

    print ("Cargando datos COSMIC desde archivos locales...")

    df_raw = cosmic_loader.load_cosmic_raw(raw_filename, classification_filename)

else:
    print ("Cargando datos del dataset de ejemplo...")
    df_raw = cosmic_loader.load_cosmic_raw(example_cosmic)

for df in df_raw:
    display(df.head())

Cargando datos COSMIC desde archivos locales...


,GENE_SYMBOL,COSMIC_GENE_ID,TRANSCRIPT_ACCESSION,COSMIC_SAMPLE_ID,SAMPLE_NAME,COSMIC_PHENOTYPE_ID,GENOMIC_MUTATION_ID,LEGACY_MUTATION_ID,MUTATION_ID,MUTATION_CDS,...,GENOME_STOP,STRAND,PUBMED_PMID,COSMIC_STUDY_ID,HGVSP,HGVSC,HGVSG,GENOMIC_WT_ALLELE,GENOMIC_MUT_ALLELE,MUTATION_SOMATIC_STATUS
0,FGFR4,COSG98910,ENST00000292408.8,COSS1783546,TCGA-EY-A212-01,COSO29025463,COSV52808563,COSM1066148,119817879,c.439C>T,...,177090940.0,+,NaN,COSU419,ENSP00000292408.4:p.Pro147Ser,ENST00000292408.8:c.439C>T,5:g.177090940C>T,C,T,Variant of unknown origin
1,FGFR4,COSG98910,ENST00000292408.8,COSS2522400,RMS2110,COSO105755343,COSV52800715,COSM3738142,119805800,c.1648G>C,...,177095550.0,+,24436047.0,NaN,ENSP00000292408.4:p.Val550Leu,ENST00000292408.8:c.1648G>C,5:g.177095550G>C,G,C,Confirmed somatic variant
2,FGFR4,COSG98910,ENST00000292408.8,COSS2634332,CHG-14-15016T,COSO29824888,COSV52807820,COSM6273757,119817908,c.1097G>T,...,177093177.0,+,NaN,COSU660,ENSP00000292408.4:p.Arg366Met,ENST00000292408.8:c.1097G>T,5:g.177093177G>T,G,T,Confirmed somatic variant
3,FGFR4,COSG98910,ENST00000292408.8,COSS1651047,TCGA-AA-3845-01,COSO28544954,COSV52807001,COSM3340981,119813804,c.2334C>T,...,177097601.0,+,22810696.0,COSU376,ENSP00000292408.4:p.Ser778=,ENST00000292408.8:c.2334C>T,5:g.177097601C>T,C,T,Confirmed somatic variant
4,FGFR4,COSG98910,ENST00000292408.8,COSS2121664,TCGA-EE-A2GN-06,COSO32666078,COSV52800464,COSM3614473,119817767,c.1048G>A,...,177092775.0,+,NaN,COSU540,ENSP00000292408.4:p.Val350Met,ENST00000292408.8:c.1048G>A,5:g.177092775G>A,G,A,Confirmed somatic variant


,COSMIC_PHENOTYPE_ID,PRIMARY_SITE,SITE_SUBTYPE_1,SITE_SUBTYPE_2,SITE_SUBTYPE_3,PRIMARY_HISTOLOGY,HISTOLOGY_SUBTYPE_1,HISTOLOGY_SUBTYPE_2,HISTOLOGY_SUBTYPE_3,NCI_CODE,EFO
0,COSO27985024,haematopoietic_and_lymphoid_tissue,NS,NS,NS,haematopoietic_neoplasm,acute_myeloid_leukaemia,M5b,NS,C3171,http://www.ebi.ac.uk/efo/EFO_0000222
1,COSO28974826,small_intestine,duodenum,NS,NS,carcinoma,adenocarcinoma,NS,NS,C7889,http://www.ebi.ac.uk/efo/EFO_1000223
2,COSO36286727,thyroid,NS,NS,NS,carcinoma,papillary_carcinoma,follicular_variant,NS,C7381,http://www.ebi.ac.uk/efo/EFO_1000261
3,COSO33006107,skin,arm,NS,NS,benign_melanocytic_nevus,NS,NS,NS,C7571,http://purl.obolibrary.org/obo/MONDO_0044794
4,COSO360710843,soft_tissue,fibrous_tissue_and_uncertain_origin,stomach,NS,GIST_tumourlet(hyalinizing_stromal_tumour;minu...,NS,NS,NS,C3868,http://purl.obolibrary.org/obo/MONDO_0011719


## 2. Filtrado y deduplicación

> **TODO:** aplicar filtros del config y reportar n inicial -> n final por filtro en una tabla. El TSV de ejemplo incluye un duplicado, una mutacion silenciosa y una variante no confirmada para probar los filtros.

In [3]:
# ============================================================
# CHUNK 2.1: Filtrado del dataset de COSMIC
# ============================================================

merged_df = reduce(lambda left, right: pd.merge
                   (left, right, on='COSMIC_PHENOTYPE_ID', how='outer'), 
                   df_raw)


filtered_df = cosmic_loader.filter_somatic_missense(merged_df, CFG['cosmic']['filters'])

filtered_dedup_df = cosmic_loader.deduplicate_by_sample(filtered_df)

df_filtered_cosmic_data = filtered_dedup_df[cosmic_loader.EXPECTED_COSMIC_COLUMNS]
df_filtered_cosmic_data.to_csv(OUT/'cosmic_filtered.csv', index=False)

print("Cargando DataFrame filtrado guardado en disco...")
display(df_filtered_cosmic_data.head())

Conteo Inicial,Mutaciones Somáticas,Mutaciones Missense,Conteo Final
66029,20028,18883,18883


Cargando DataFrame filtrado guardado en disco...


,GENE_SYMBOL,MUTATION_AA,MUTATION_SOMATIC_STATUS,MUTATION_DESCRIPTION,COSMIC_SAMPLE_ID,PRIMARY_SITE,PRIMARY_HISTOLOGY
99,KRAS,p.G12D,Confirmed somatic variant,missense_variant,COSS1728521,soft_tissue,rhabdomyosarcoma
416,HRAS,p.A59T,Confirmed somatic variant,missense_variant,COSS2911403,soft_tissue,angiosarcoma
438,HRAS,p.G13V,Confirmed somatic variant,missense_variant,COSS1809247,soft_tissue,angiosarcoma
501,KRAS,p.P140S,Confirmed somatic variant,missense_variant,COSS2215317,peritoneum,other
502,KRAS,p.G12D,Confirmed somatic variant,missense_variant,COSS2215317,peritoneum,other


## 3. Construcción del DataFrame curado por gen

La salida debe cumplir el contrato de `data/processed/cosmic_curated.csv`. Ver `docs/contratos_datos.md`.

In [6]:
# ============================================================
# CHUNK 3.1: Construcción del DataFrame curado
# ============================================================

df_curated_cosmic_data = cosmic_loader.build_curated_dataset(df_filtered_cosmic_data, CFG['cosmic']['version'])
df_curated_cosmic_data.to_csv(OUT/'cosmic_curated.csv', index=False)

print("Cargando DataFrame curado guardado en disco...")
print(f"Número de mutaciones en el dataset curado: {len(df_curated_cosmic_data)} mutaciones")
display(df_curated_cosmic_data.groupby('gene').head(5))

Cargando DataFrame curado guardado en disco...
Número de mutaciones en el dataset curado: 469 mutaciones


,gene,uniprot_id,position,wt_aa,mut_aa,hgvs_p,sample_count,tumor_types,primary_tissues,cosmic_version
149,KRAS,P01116,12,G,D,p.G12D,4697,Ewing_sarcoma-peripheral_primitive_neuroectode...,NS | autonomic_ganglia | biliary_tract | bone ...,v104
156,KRAS,P01116,12,G,V,p.G12V,3625,Wilms_tumour | aberrant_crypt_foci | adenoma |...,NS | autonomic_ganglia | biliary_tract | breas...,v104
160,KRAS,P01116,13,G,D,p.G13D,1716,NS | aberrant_crypt_foci | adenoma | angiosarc...,NS | biliary_tract | breast | cervix | endomet...,v104
148,KRAS,P01116,12,G,C,p.G12C,1345,aberrant_crypt_foci | adenoma | angiosarcoma |...,NS | biliary_tract | breast | cervix | endomet...,v104
147,KRAS,P01116,12,G,A,p.G12A,728,aberrant_crypt_foci | adenoma | carcinoid-endo...,NS | autonomic_ganglia | biliary_tract | breas...,v104
20,HRAS,P01112,13,G,R,p.G13R,240,adnexal_tumour | benign_melanocytic_nevus | ca...,adrenal_gland | autonomic_ganglia | breast | l...,v104
58,HRAS,P01112,61,Q,R,p.Q61R,214,angiosarcoma | benign_melanocytic_nevus | carc...,adrenal_gland | autonomic_ganglia | breast | g...,v104
56,HRAS,P01112,61,Q,K,p.Q61K,84,adenoma-nodule-goitre | carcinoma | germ_cell_...,adrenal_gland | autonomic_ganglia | bone | bre...,v104
14,HRAS,P01112,12,G,S,p.G12S,64,carcinoma | malignant_melanoma | other | rhabd...,bone | endometrium | large_intestine | oesopha...,v104
57,HRAS,P01112,61,Q,L,p.Q61L,64,angiosarcoma | benign_melanocytic_nevus | carc...,breast | central_nervous_system | liver | lung...,v104


## 4. Descarga de estructuras PDB

Usa las estructuras principales declaradas en `configs/config.yaml`. Guarda los PDBs en `data/external/pdb/` y valida que Biopython puede parsearlos.

In [5]:
# ============================================================
# CHUNK 4.1: Descarga de estructuras PDB
# ============================================================

EXTERNAL = ROOT / CFG['paths']['data_external'] / 'pdb'

pdbl = PDBList()
parser = PDBParser(QUIET=True)

ras_genes = cosmic_loader.RAS_GENES

for gene in ras_genes:

    gene_cfg = CFG['family']['members'][gene]

    pdb_ids = (
        gene_cfg['pdbs']['primary']['id'],
        gene_cfg['pdbs']['secondary']['id']
    )

    for pdb_id in pdb_ids:

        pdb_file = pdbl.retrieve_pdb_file(
            pdb_id,
            pdir=EXTERNAL,
            file_format='pdb'
        )

        structure = parser.get_structure(
            pdb_id,
            pdb_file
        )

        print(f'{gene} - {pdb_id}: parse OK')

Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb4obe.ent' 
KRAS - 4OBE: parse OK
Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb5uk9.ent' 
KRAS - 5UK9: parse OK
Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb3k8y.ent' 
HRAS - 3K8Y: parse OK
Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb2rge.ent' 
HRAS - 2RGE: parse OK
Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb5uhv.ent' 
NRAS - 5UHV: parse OK
Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb3con.ent' 
NRAS - 3CON: parse OK


---
## Resumen del Hito 1

En este notebook hemos completado el **Hito 1** del pipeline de bioinformática estructural del TFM:

| Paso | Función utilizada | Estado |
|------|-------------------|--------|
| Carga de configuración | `load_config()` | Completado |
| Descarga de PDBs de referencia | `pdbl.retrieve_pdb_file()` | Completado |
| Carga de datos COSMIC | `load_cosmic_raw()` | Completado |
| Filtrado: somáticas missense | `filter_somatic_missense()` | Completado |
| Deduplicación por muestra | `deduplicate_by_sample()` | Completado |
| Curación del Dataset | `build_curated_dataset()` | Completado |
| Guardado del dataset curado | `df_curated.to_csv()` | Completado |

### Salida principal del Hito 1

**`data/processed/cosmic_curated.csv`**: dataset curado con mutaciones somáticas missense de KRAS, HRAS y NRAS, con columnas estandarizadas, metadatos clínicos y conteos de muestras. Es la entrada para los hitos estructurales y de machine learning.